<a href="https://colab.research.google.com/github/gerardagustina/breast-cancer-detection/blob/main/notebooks/03_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Breast Cancer Detection — 03. Model & Training
**Goal:** Build a DenseNet121 classifier with transfer learning, train it and evaluate it.

In [2]:
!pip install -q medmnist monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 11.3 MB/s eta 0:00:00


In [3]:
import medmnist, torch
from medmnist import BreastMNIST
from monai.data import set_track_meta, ArrayDataset, DataLoader
from monai.transforms import (Compose, EnsureChannelFirst, ScaleIntensityRange,
                              RandFlip, RandRotate, RandZoom)
set_track_meta(False)

splits = {s: BreastMNIST(split=s, download=True, size=224) for s in ["train", "val", "test"]}

base = [EnsureChannelFirst(channel_dim="no_channel"),
        ScaleIntensityRange(a_min=0, a_max=255, b_min=0, b_max=1, clip=True)]
train_tf = Compose(base + [RandFlip(prob=0.5, spatial_axis=1),
                           RandRotate(range_x=0.26, prob=0.5),
                           RandZoom(min_zoom=0.9, max_zoom=1.1, prob=0.5)])
val_tf = Compose(base)

def make_ds(split, tf):
    d = splits[split]
    return ArrayDataset(img=d.imgs, img_transform=tf, labels=d.labels.ravel())

train_loader = DataLoader(make_ds("train", train_tf), batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(make_ds("val",   val_tf),   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(make_ds("test",  val_tf),   batch_size=32, shuffle=False, num_workers=2)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Datos listos ✅ | device:", device)

100%|██████████| 30.9M/30.9M [00:51<00:00, 599kB/s]


Datos listos ✅ | device: cuda


In [4]:
import torch.nn as nn
from torchvision.models import densenet121, DenseNet121_Weights

class BreastNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)   # pesos de ImageNet
        self.net.classifier = nn.Linear(self.net.classifier.in_features, 1)  # 1000 clases → 1 salida

    def forward(self, x):
        x = x.repeat(1, 3, 1, 1)          # gris (1 canal) → 3 canales, como espera ImageNet
        return self.net(x).squeeze(1)     # (32, 1) → (32,)

model = BreastNet().to(device)

print(f"Pesos totales: {sum(p.numel() for p in model.parameters()):,}")
x, y = next(iter(train_loader))
print("Entrada:", x.shape, "→ Salida:", model(x.to(device)).shape)

Pesos totales: 6,954,881
Entrada: torch.Size([32, 1, 224, 224]) → Salida: torch.Size([32])


## Model architecture
- **DenseNet121** (torchvision) initialised with **ImageNet weights** (transfer learning).
- Classifier head replaced: 1000 ImageNet classes → **1 output (logit)**.
- Grayscale input replicated to 3 channels to match the pretrained network.
- **6.95M parameters.** Dense connectivity enables feature reuse with few parameters,
  reducing overfitting on a small dataset (546 training images).